# Importing Packages

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.ensemble import HistGradientBoostingClassifier

## Importing our data

In [2]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/processed/cleaned_train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/processed/cleaned_train.csv'
    },
    'test': {
        'local': '../data/processed/cleaned_test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/processed/cleaned_test.csv'
    },
    'sample_sub': {
            'local': '../data/raw/SampleSubmission.csv',
            'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
        }
    
}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
sample_sub = data['sample_sub']

Loaded train from local path.
Loaded test from local path.
Loaded sample_sub from local path.


## Setting up our columns

In [3]:
target_col = 'cost_category'
id_col = 'Tour_ID'
target_classes = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']

## Preparing our categorical columns

In [ ]:
train_df = train
test_df = test

cat_cols = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity', 
    'info_source', 'tour_arrangement', 'package_transport_int', 
    'package_accomodation', 'package_food', 'package_transport_tz', 
    'package_sightseeing', 'package_guided_tour', 'package_insurance', 'first_trip_tz'
]

X = train_df.drop(columns=[id_col, target_col])
y = train_df[target_col]
X_test = test_df.drop(columns=[id_col])


# Convert text/object columns to pandas 'category' type for gradient boosting
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

## Stratified Cross-Validation & Modeling

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(train_df), len(target_classes)))
test_preds = np.zeros((len(test_df), len(target_classes)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # Classifier native to categorical feature handling
    model = HistGradientBoostingClassifier(
        categorical_features=cat_cols,
        max_iter=300,
        learning_rate=0.05,
        random_state=42
    )
    
    model.fit(X_tr, y_tr)
    
    # Out-of-fold validation prediction
    oof_preds[val_idx] = model.predict_proba(X_va)
    # Average test set predictions across folds
    test_preds += model.predict_proba(X_test) / skf.n_splits

# Print Out-of-Fold validation score
cv_loss = log_loss(y, oof_preds)
print(f"\n==========================================")
print(f"Overall OOF Log Loss: {cv_loss:.5f}")
print(f"==========================================")

## Format & Export Submission File

In [ ]:
# Map probability array back to class columns matching sample submission
submission = pd.DataFrame(test_preds, columns=model.classes_)
submission.insert(0, id_col, test_df[id_col])

In [ ]:
# Strictly match SampleSubmission.csv column order
target_order = ['Tour_ID', 'High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']
submission = submission[target_order]

In [ ]:
#Viewing submission in notebook
submission

In [ ]:
# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Saved final_submission.csv successfully!")